# ประเมินโอกาสมีผู้เสียชีวิตจากอุบัติเหตุทางถนนในกรุงเทพมหานคร
### Road Accident Fatality Classification

## Problem Framing

โปรเจกต์นี้ใช้ข้อมูล **เฉพาะกรุงเทพมหานคร ปี 2020–2026** เพื่อประเมินว่า **หากเกิดอุบัติเหตุภายใต้เงื่อนไขที่ระบุ จะมีโอกาสเท่าไรที่เหตุนั้นมีผู้เสียชีวิตอย่างน้อยหนึ่งคน** ผู้ใช้กรอก 4 ค่า ได้แก่ รหัสสายทาง กิโลเมตร วันเวลา และสภาพอากาศ

### เตรียม Python

ใน VS Code ให้กด **Select Kernel → Python Environments** แล้วเลือก Python ที่มี `ipykernel` จากนั้นกด **Run All** เซลล์ด้านล่างจะติดตั้งแพ็กเกจที่ยังขาดใน Python ตัวที่เลือก

In [100]:
import importlib.util
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')

# ชื่อด้านซ้ายใช้ import ส่วนชื่อด้านขวาใช้ติดตั้งด้วย pip
required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'joblib': 'joblib',
}
# ตรวจทีละแพ็กเกจและติดตั้งเฉพาะตัวที่ยังไม่มีใน Python ที่กำลังรัน Notebook
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

install_error = None
if missing_packages:
    # sys.executable ทำให้ pip ติดตั้งลงใน Python ตัวเดียวกับ Kernel ที่เลือกไว้
    command = [
        sys.executable, '-m', 'pip', 'install',
        '--quiet', '--disable-pip-version-check',
        *missing_packages,
    ]
    completed = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    if completed.returncode != 0:
        install_error = completed.stderr.strip()
    importlib.invalidate_caches()

# ลอง import จริงอีกครั้ง เพราะ VS Code อาจติดตั้งเสร็จพร้อมกับที่เซลล์นี้รันอยู่
for module_name in required_packages:
    try:
        importlib.import_module(module_name)
    except (ImportError, OSError) as error:
        if install_error and 'WinError 32' in install_error:
            raise RuntimeError(
                'Windows กำลังใช้งานไฟล์ไลบรารีอยู่ ให้รอจน VS Code ติดตั้งเสร็จ '
                'แล้วกด Restart Kernel และ Run All'
            ) from error
        if install_error:
            raise RuntimeError('ติดตั้งไลบรารีไม่สำเร็จ:\n' + install_error) from error
        raise ImportError(f'เปิดใช้ไลบรารี {module_name} ไม่สำเร็จ') from error

print('พร้อมใช้งาน: ติดตั้งและตรวจสอบไลบรารีที่จำเป็นครบแล้ว')

พร้อมใช้งาน: ติดตั้งและตรวจสอบไลบรารีที่จำเป็นครบแล้ว


## Data Acquisition — เตรียมไฟล์ CSV

หากเปิดใน Google Colab ให้รันเซลล์นี้แล้วเลือกไฟล์ CSV ที่ยังไม่ได้อัปโหลด
หากเปิดใน VS Code/Jupyter ให้วางไฟล์ทั้ง 7 ไว้ในโฟลเดอร์เดียวกับ Notebook

Notebook จะเลือกเฉพาะแถวที่ระบุจังหวัดเป็นกรุงเทพมหานครจากทุกไฟล์

- `accident2020.csv`
- `accident2021.csv`
- `accident2022.csv`
- `accident2023.csv`
- `accident2024.csv`
- `accident2025.csv`
- `accident2026.csv`

In [101]:
from pathlib import Path

# เก็บชื่อไฟล์คู่กับปี เพื่อให้ใช้ชื่อเดียวกันตอนตรวจไฟล์และตอนโหลดข้อมูล
# กำหนดช่วงฝึก ปีปรับเทียบ และปีทดสอบไว้ก่อนอ่านข้อมูล เพื่อใช้ช่วงเดียวกันตลอด Notebook
target_province = 'กรุงเทพมหานคร'
train_start_year = 2020
train_end_year = 2024
validation_year = 2025
test_year = 2026
data_years = range(train_start_year, test_year + 1)
selected_files = {year: f'accident{year}.csv' for year in data_years}
missing_files = [name for name in selected_files.values() if not Path(name).is_file()]

if missing_files:
    # Colab อัปโหลดไฟล์ได้จากหน้าจอ ส่วน VS Code อ่านจากโฟลเดอร์ของ Notebook
    try:
        from google.colab import files
    except ImportError:
        pass
    else:
        print('กรุณาอัปโหลดไฟล์ CSV:', ', '.join(missing_files))
        files.upload()
        missing_files = [name for name in missing_files if not Path(name).is_file()]

if missing_files:
    raise FileNotFoundError(
        f'ไม่พบไฟล์ CSV: {", ".join(missing_files)}. '
        'ให้วางไฟล์ในโฟลเดอร์เดียวกับ Notebook หรืออัปโหลดใน Colab'
    )
print('พบไฟล์ข้อมูลครบแล้ว:', selected_files)

พบไฟล์ข้อมูลครบแล้ว: {2020: 'accident2020.csv', 2021: 'accident2021.csv', 2022: 'accident2022.csv', 2023: 'accident2023.csv', 2024: 'accident2024.csv', 2025: 'accident2025.csv', 2026: 'accident2026.csv'}


## Data Preparation — เตรียมข้อมูลและสร้าง Target

| ข้อมูลที่ผู้ใช้กรอก | ตัวแปรที่โมเดลใช้ |
|---|---|
| รหัสสายทาง | รหัสสายทาง |
| กิโลเมตร | ก.ม. |
| วันที่และเวลา | เดือน วันในสัปดาห์ และชั่วโมง |
| สภาพอากาศ | สภาพอากาศ |

**Target `fatal`** มีค่า 1 เมื่อมีผู้เสียชีวิตอย่างน้อยหนึ่งคน และมีค่า 0 เมื่อไม่มีผู้เสียชีวิต จำนวนผู้เสียชีวิตใช้สร้าง Target เท่านั้น

### ทำชื่อคอลัมน์ให้ตรงกัน

In [102]:
import re

import numpy as np
import pandas as pd

# ไฟล์แต่ละปีอาจตั้งชื่อต่างกัน เช่น KM กับ ก.ม. ชื่อด้านซ้ายคือชื่อกลางที่ใช้ต่อใน Notebook
FEATURE_ALIASES = {
    'วันที่เกิดเหตุ': ['วันที่เกิดเหตุ', 'วันที่เกิดอุบัติเหตุ', 'วันที่'],
    'เวลา': ['เวลา', 'เวลาเกิดเหตุ', 'เวลาที่เกิดเหตุ'],
    'จำนวนผู้เสียชีวิต': ['จำนวนผู้เสียชีวิต', 'ผู้เสียชีวิต'],
    'รหัสสายทาง': ['รหัสสายทาง', 'รหัสถนน', 'หมายเลขทางหลวง', 'หมายเลขสายทาง'],
    'ก.ม.': ['ก.ม.', 'กม.', 'กิโลเมตร', 'หลักกิโลเมตร', 'KM'],
    'สภาพอากาศ': ['สภาพอากาศ', 'สภาพอากาศขณะเกิดเหตุ', 'อากาศ'],
}


def harmonize_columns(data, year):
    # ลบช่องว่างเกินและอักขระ BOM จากหัวคอลัมน์ก่อนจับคู่ชื่อ
    data = data.copy()
    data.columns = [
        re.sub(r'\s+', ' ', str(column).replace('\ufeff', '')).strip()
        for column in data.columns
    ]

    for standard_name, aliases in FEATURE_ALIASES.items():
        available = [name for name in aliases if name in data.columns]
        if not available:
            data[standard_name] = pd.NA
            continue

        # ใช้ชื่อแรกตามลำดับในรายการ แล้วเติมเฉพาะแถวที่ว่างจากชื่ออื่นที่พบ
        values = data[available[0]].replace(r'^\s*$', pd.NA, regex=True)
        for name in available[1:]:
            extra_values = data[name].replace(r'^\s*$', pd.NA, regex=True)
            values = values.fillna(extra_values)
        data[standard_name] = values

    # วันที่ เวลา และจำนวนผู้เสียชีวิตต้องมีข้อมูล เพื่อใช้สร้างตัวแปรและคำตอบจริง
    required = ['วันที่เกิดเหตุ', 'เวลา', 'จำนวนผู้เสียชีวิต']
    missing = [name for name in required if data[name].isna().all()]
    if missing:
        raise ValueError(f'ข้อมูลปี {year} ไม่มีคอลัมน์ที่จำเป็น: {missing}')
    return data

### แปลงวันที่และเวลาให้อยู่ในรูปแบบเดียวกัน

In [103]:
def parse_date(series, year):
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, errors='coerce')

    # ไฟล์ 2024–2025 มีวันที่แบบเลข Excel เช่น 45658 หมายถึง 1 ม.ค. 2025
    excel_numbers = pd.to_numeric(series, errors='coerce')
    excel_dates = pd.to_datetime(excel_numbers, unit='D', origin='1899-12-30', errors='coerce')

    # เลือกเฉพาะวันที่ที่เป็นข้อความเพื่อแปลงแยกจากเลข Excel
    text = series.where(excel_numbers.isna()).astype('string').str.strip()
    has_slashes = text.str.fullmatch(r'[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}', na=False)

    # ไฟล์ 2020–2023 ใช้เดือน/วัน/ปี ส่วน 2026 ใช้วัน/เดือน/ปี วันที่ 3/4 จึงตีความต่างกัน
    date_format = '%d/%m/%Y' if year == 2026 else '%m/%d/%Y'
    slash_dates = pd.to_datetime(text.where(has_slashes), format=date_format, errors='coerce')
    other_dates = pd.to_datetime(text.where(~has_slashes), errors='coerce')

    # รวมผลจากแต่ละรูปแบบ ค่าใดอ่านไม่ได้จะยังเป็น NaT เพื่อจัดการตอนเตรียมข้อมูล
    return slash_dates.fillna(other_dates).fillna(excel_dates)


def parse_hour(value):
    if pd.isna(value):
        return np.nan

    # เวลา 13:30 เป็น 13.5 ชั่วโมง จึงเก็บรายละเอียดของนาทีไว้
    if hasattr(value, 'hour'):
        return value.hour + value.minute / 60

    # เวลา Excel ที่เป็นเศษส่วนของวัน เช่น 0.5 คูณ 24 จะได้เวลา 12:00 น.
    if isinstance(value, (float, int, np.number)):
        number = float(value)
        return number * 24 if 0 <= number < 1 else number

    try:
        parts = str(value).split(':')
        hours = int(parts[0])
        minutes = int(parts[1])
        return hours + minutes / 60
    except (IndexError, ValueError):
        return np.nan

### กรองกรุงเทพมหานคร แล้วสร้าง Features และ Target

In [104]:
# อ่านทีละปีและปรับวันเวลาก่อนรวม เพราะไฟล์แต่ละปีใช้รูปแบบวันที่ต่างกัน
frames = []
for year, filename in selected_files.items():
    data = pd.read_csv(filename, encoding='utf-8-sig')

    # ใช้ขอบเขตจังหวัดเดียวกันทุกปี แถวที่ไม่ระบุจังหวัดจะไม่อยู่ในชุดกรุงเทพฯ
    province = data['จังหวัด'].astype('string').str.strip()
    data = data.loc[province.eq(target_province).fillna(False)].copy()
    if data.empty:
        raise ValueError(f'ไม่พบข้อมูลจังหวัด {target_province} ในไฟล์ {filename}')

    data = harmonize_columns(data, year)
    data['date'] = parse_date(data['วันที่เกิดเหตุ'], year)
    data['hour'] = data['เวลา'].map(parse_hour)
    data['year'] = year
    frames.append(data)

df = pd.concat(frames, ignore_index=True)

# วันเวลา 1 ช่องถูกแยกเป็นเดือน วันในสัปดาห์ (จันทร์ = 0) และชั่วโมงทศนิยม
df['month'] = df['date'].dt.month
df['weekday'] = df['date'].dt.dayofweek

# รหัสสายทางและสภาพอากาศเป็นหมวดหมู่ แม้รหัสสายทางบางค่าจะเขียนด้วยตัวเลข
CATEGORICAL_FEATURES = ['รหัสสายทาง', 'สภาพอากาศ']
NUMERIC_FEATURES = ['ก.ม.', 'month', 'weekday', 'hour']
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

for column in CATEGORICAL_FEATURES:
    values = df[column].replace(r'^\s*$', pd.NA, regex=True)
    df[column] = values.fillna('ไม่ระบุ').astype(str).str.strip()

# รหัสที่ CSV อ่านเป็น 3.0 และรหัส 3 ต้องอยู่ในหมวดเดียวกัน
df['รหัสสายทาง'] = df['รหัสสายทาง'].str.replace(r'^(\d+)\.0$', r'\1', regex=True)

for column in NUMERIC_FEATURES:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# ข้ามแถวที่ยังไม่รู้วันเกิดเหตุหรือคำตอบจริง แล้วให้ 1 หมายถึงมีผู้เสียชีวิตอย่างน้อยหนึ่งคน
df['death_count'] = pd.to_numeric(df['จำนวนผู้เสียชีวิต'], errors='coerce')
df = df.dropna(subset=['date', 'death_count']).copy()
df['fatal'] = (df['death_count'] > 0).astype(int)

# จำนวนเหตุเสียชีวิตคือผลรวมของ fatal สัดส่วนเหตุเสียชีวิตคือค่าเฉลี่ยของ fatal
summary = df.groupby('year')['fatal'].agg(['count', 'sum', 'mean'])
summary.columns = ['จำนวนเหตุการณ์', 'เหตุเสียชีวิต', 'สัดส่วนเสียชีวิต']
print(f'ข้อมูล{target_province}ทั้งหมด: {len(df):,} เหตุการณ์')
print(summary)

ข้อมูลกรุงเทพมหานครทั้งหมด: 15,289 เหตุการณ์
      จำนวนเหตุการณ์  เหตุเสียชีวิต  สัดส่วนเสียชีวิต
year                                                 
2020            1689             24          0.014210
2021            1199             28          0.023353
2022            1723             26          0.015090
2023            2668             67          0.025112
2024            2917             45          0.015427
2025            2865             59          0.020593
2026            2228             42          0.018851


## Exploratory Data Analysis — สำรวจข้อมูล

กราฟแสดงสัดส่วนอุบัติเหตุที่มีผู้เสียชีวิตในกรุงเทพมหานครของแต่ละปี ตัวเลขเหนือแท่งมาจากจำนวนเหตุที่มีผู้เสียชีวิตหารด้วยอุบัติเหตุทั้งหมดของปีนั้น เส้นประคือสัดส่วนรวมของข้อมูลกรุงเทพมหานครทั้งเจ็ดปี

### ฟังก์ชันสำหรับวาดกราฟ

ส่วนนี้กำหนดหน้าตากราฟไว้ที่เดียว เซลล์วิเคราะห์ด้านล่างส่งค่าจากตารางเข้าไปวาดได้โดยตรง

In [105]:
from html import escape

from IPython.display import HTML, Markdown, display

def show_grouped_bars(title, values, ticks, x_label, y_label, colors, reference=None, reference_label=''):
    """วาดกราฟแท่งแนวตั้งบนพื้นขาวพร้อมเส้นตารางและค่าบนแท่ง"""
    left, right, top, bottom = 70, 866, 55, 322
    upper = ticks[-1]

    def plot_y(value):
        return bottom - value / upper * (bottom - top)

    # วาดแกนและเส้นตารางก่อน เพื่อให้แท่งและป้ายค่าอยู่ด้านหน้า
    marks = [
        '<rect width="900" height="410" fill="#ffffff"/>',
        f'<rect x="{left}" y="{top}" width="{right-left}" height="{bottom-top}" fill="#ffffff" stroke="#cccccc"/>',
        f'<text x="450" y="30" text-anchor="middle" font-size="19">{escape(title)}</text>',
        f'<text x="468" y="378" text-anchor="middle">{escape(x_label)}</text>',
        f'<text x="19" y="190" text-anchor="middle" transform="rotate(-90 19 190)">{escape(y_label)}</text>',
    ]
    for tick in ticks:
        y = plot_y(tick)
        marks.append(f'<line x1="{left}" x2="{right}" y1="{y}" y2="{y}" stroke="#cccccc"/>')
        marks.append(f'<text x="{left-8}" y="{y+5}" text-anchor="end">{tick:.0%}</text>')

    bar_labels = []
    # หนึ่งกลุ่มแทนหนึ่งปีหรือหนึ่งโมเดล แต่ละแท่งในกลุ่มแทนตัวชี้วัดหนึ่งค่า
    group_width = (right - left) / len(values)
    bar_width = min(76, group_width * 0.78 / len(values.columns))
    for group, (label, row) in enumerate(values.iterrows()):
        center = left + group_width * (group + 0.5)
        marks.append(f'<line x1="{center}" x2="{center}" y1="{top}" y2="{bottom}" stroke="#eeeeee"/>')
        marks.append(f'<text x="{center}" y="{bottom+23}" text-anchor="middle">{escape(str(label))}</text>')
        for series, (metric, color) in enumerate(zip(values.columns, colors)):
            value = row[metric]
            x = center + (series - len(values.columns) / 2) * bar_width
            y = plot_y(value)
            label = f'{value:.1%}'
            marks.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{bar_width-2:.2f}" height="{bottom-y:.2f}" fill="{color}"><title>{escape(str(metric))} {label}</title></rect>')
            label_style = f'x="{x+bar_width/2:.2f}" y="{y-6:.2f}" text-anchor="middle" font-size="11"'
            bar_labels.append(f'<text {label_style} stroke="#fff" stroke-width="3" aria-hidden="true">{label}</text>')
            bar_labels.append(f'<text {label_style}>{label}</text>')

    # วาดเส้นประเมื่อกราฟมีค่าอ้างอิง เช่น สัดส่วนเหตุเสียชีวิตรวมของทุกปี
    if reference is not None:
        y = plot_y(reference)
        marks.append(f'<line x1="{left}" x2="{right}" y1="{y}" y2="{y}" stroke="#777777" stroke-width="2" stroke-dasharray="7 5"/>')
        marks.append('<line x1="86" x2="111" y1="73" y2="73" stroke="#777777" stroke-width="2" stroke-dasharray="7 5"/>')
        marks.append(f'<text x="117" y="78">{escape(reference_label)} {reference:.1%}</text>')
    if len(values.columns) > 1:
        for index, (metric, color) in enumerate(zip(values.columns, colors)):
            x = 190 + index * 145
            marks.append(f'<rect x="{x}" y="394" width="19" height="9" fill="{color}"/>')
            marks.append(f'<text x="{x+25}" y="403">{escape(str(metric))}</text>')

    display(HTML(
        '<div style="max-width:900px">'
        f'<svg viewBox="0 0 900 410" role="img" aria-label="{escape(title)}" '
        'font-family="Tahoma" font-size="14" fill="#222222" style="display:block;width:100%;height:auto;background:#fff;font-family:Tahoma,Leelawadee UI,Arial,sans-serif">'
        + ''.join(marks + bar_labels) + '</svg></div>'
    ))


def show_confusion_matrix(counts):
    """วาดตารางจริงเทียบกับคำทำนาย โดยใช้สเกลสีเดียวกันทั้งสี่ช่อง"""
    max_count = max(max(row) for row in counts)
    labels = ('ไม่มีผู้เสียชีวิต', 'มีผู้เสียชีวิต')
    heatmap = [
        '<rect width="540" height="410" fill="#ffffff"/>',
        '<text x="270" y="28" text-anchor="middle" font-size="19">Confusion Matrix — Logistic Regression</text>',
        '<text x="276" y="384" text-anchor="middle">Predicted label</text>',
        '<text x="18" y="190" transform="rotate(-90 18 190)" text-anchor="middle">True label</text>',
        '<defs><linearGradient id="fatal-count-scale" x1="0" y1="1" x2="0" y2="0">'
        '<stop offset="0%" stop-color="#fff5f0"/><stop offset="100%" stop-color="#99000d"/></linearGradient></defs>',
        '<rect x="445" y="55" width="19" height="280" fill="url(#fatal-count-scale)" stroke="#cccccc"/>',
    ]
    for index, label in enumerate(labels):
        heatmap.append(f'<text x="{196 + index*140}" y="357" text-anchor="middle">{label}</text>')
        heatmap.append(f'<text x="116" y="{129 + index*140}" text-anchor="end">{label}</text>')
    for row_index, row in enumerate(counts):
        for col_index, count in enumerate(row):
            strength = count / max_count
            # ใช้สเกลสีเดียวกันทั้งสี่ช่อง และปรับสีตัวเลขให้อ่านง่าย
            shade = [round(light + (dark - light) * strength) for light, dark in zip((255, 245, 240), (153, 0, 13))]
            fill = '#' + ''.join(f'{channel:02x}' for channel in shade)
            color = '#ffffff' if strength > 0.55 else '#5c1113'
            x, y = 126 + col_index * 140, 55 + row_index * 140
            heatmap.append(f'<rect x="{x}" y="{y}" width="140" height="140" fill="{fill}" stroke="#dddddd"/>')
            heatmap.append(f'<text x="{x+70}" y="{y+77}" text-anchor="middle" font-size="21" fill="{color}">{count:,}</text>')
    for position in (0, 0.25, 0.5, 0.75, 1):
        y = 335 - position * 280
        heatmap.append(f'<line x1="464" x2="469" y1="{y}" y2="{y}" stroke="#555555"/>')
        heatmap.append(f'<text x="474" y="{y+4}">{max_count*position:,.0f}</text>')

    display(HTML(
        '<div style="max-width:540px">'
        '<svg viewBox="0 0 540 410" role="img" aria-label="Confusion Matrix ของ Logistic Regression" '
        'font-family="Tahoma" font-size="14" fill="#222222" style="display:block;width:100%;height:auto;background:#fff;font-family:Tahoma,Leelawadee UI,Arial,sans-serif">'
        + ''.join(heatmap) + '</svg></div>'
    ))

In [106]:
# ค่าเฉลี่ยของ Target แบบ 0/1 เท่ากับสัดส่วนเหตุที่มีผู้เสียชีวิต
yearly = df.groupby('year').agg(
    accidents=('fatal', 'size'),
    fatal_accidents=('fatal', 'sum'),
    fatal_rate=('fatal', 'mean'),
    last_date=('date', 'max'),
)
yearly_plot = yearly[['fatal_rate']].rename(columns={'fatal_rate': 'สัดส่วนเหตุเสียชีวิต'})
yearly_plot.index = [
    str(year) if (last_date.month, last_date.day) == (12, 31) else f'{year} ถึง {last_date:%d/%m}'
    for year, last_date in yearly['last_date'].items()
]
# ใช้แกนทีละ 1 จุดเปอร์เซ็นต์ เพื่อให้เห็นความต่างในข้อมูลกรุงเทพฯ
axis_limit = max(0.04, np.ceil(yearly['fatal_rate'].max() * 100 + 1) / 100)
show_grouped_bars(
    'สัดส่วนอุบัติเหตุที่มีผู้เสียชีวิตรายปีในกรุงเทพมหานคร',
    yearly_plot,
    np.arange(round(axis_limit * 100) + 1) / 100,
    'ปีที่เกิดเหตุ', 'สัดส่วนเหตุเสียชีวิต', ['#d9534f'],
    reference=df['fatal'].mean(), reference_label='สัดส่วนรวม',
)

# เทียบช่วงวันที่เดียวกัน เพราะข้อมูลปีล่าสุดอาจยังไม่ครบปี
last_date = yearly['last_date'].iloc[-1]
last_day = last_date.month * 100 + last_date.day
same_period = df.loc[(df['date'].dt.month * 100 + df['date'].dt.day) <= last_day]
rates = same_period.groupby('year')['fatal'].mean()
comparison = ' · '.join(f'{year} {rate:.1%}' for year, rate in rates.items())
display(Markdown(
    f"**อ่านกราฟ** เมื่อเทียบช่วงเดียวกันตั้งแต่ 1 ม.ค. ถึง {last_date:%d/%m} ของทุกปี สัดส่วนคือ {comparison}\n\n"
    "สัดส่วนนี้คำนวณจากอุบัติเหตุที่บันทึกไว้ จำนวนเที่ยวเดินทางหรือปริมาณรถเป็นข้อมูลอีกส่วนหนึ่งที่จำเป็นต่อการวัดความเสี่ยงต่อการเดินทาง"
))

# ใช้เฉพาะชุดฝึกเพื่อสำรวจความสัมพันธ์ระหว่างช่วงเวลากับสัดส่วนเหตุเสียชีวิต
training_events = df.loc[df['year'].between(train_start_year, train_end_year)]
afternoon = training_events.loc[training_events['hour'].between(12, 18, inclusive='left'), 'fatal']
evening = training_events.loc[training_events['hour'].between(18, 24, inclusive='left'), 'fatal']
if not afternoon.empty and not evening.empty:
    display(Markdown(
        f"**ข้อสังเกตตามช่วงเวลาในชุดฝึกปี {train_start_year}–{train_end_year}** ช่วง **18:00–23:59** "
        f"มีสัดส่วนเหตุเสียชีวิต **{evening.mean():.1%}** ({evening.sum():,} จาก {evening.size:,} เหตุ) "
        f"เทียบกับ **{afternoon.mean():.1%}** ({afternoon.sum():,} จาก {afternoon.size:,} เหตุ) "
        "ในช่วง **12:00–17:59** ตัวเลขนี้มาจากอุบัติเหตุที่บันทึกไว้ในแต่ละช่วงเวลา"
    ))

# แสดงจำนวนจริงคู่กับเปอร์เซ็นต์ เพื่อให้เห็นว่าแต่ละปีมีเหตุเสียชีวิตกี่เหตุ
yearly_display = yearly[['accidents', 'fatal_accidents', 'fatal_rate']].copy()
yearly_display['fatal_rate'] = yearly_display['fatal_rate'].map(lambda rate: f'{rate:.1%}')
yearly_display.columns = ['อุบัติเหตุในกรุงเทพฯ', 'เหตุที่มีผู้เสียชีวิต', 'สัดส่วนเหตุเสียชีวิต']
display(yearly_display)


**อ่านกราฟ** เมื่อเทียบช่วงเดียวกันตั้งแต่ 1 ม.ค. ถึง 31/10 ของทุกปี สัดส่วนคือ 2020 1.4% · 2021 2.2% · 2022 1.7% · 2023 2.6% · 2024 1.6% · 2025 2.0% · 2026 1.9%

สัดส่วนนี้คำนวณจากอุบัติเหตุที่บันทึกไว้ จำนวนเที่ยวเดินทางหรือปริมาณรถเป็นข้อมูลอีกส่วนหนึ่งที่จำเป็นต่อการวัดความเสี่ยงต่อการเดินทาง

**ข้อสังเกตตามช่วงเวลาในชุดฝึกปี 2020–2024** ช่วง **18:00–23:59** มีสัดส่วนเหตุเสียชีวิต **2.0%** (55 จาก 2,788 เหตุ) เทียบกับ **0.9%** (27 จาก 2,934 เหตุ) ในช่วง **12:00–17:59** ตัวเลขนี้มาจากอุบัติเหตุที่บันทึกไว้ในแต่ละช่วงเวลา

,อุบัติเหตุในกรุงเทพฯ,เหตุที่มีผู้เสียชีวิต,สัดส่วนเหตุเสียชีวิต
year,,,
2020,1689,24,1.4%
2021,1199,28,2.3%
2022,1723,26,1.5%
2023,2668,67,2.5%
2024,2917,45,1.5%
2025,2865,59,2.1%
2026,2228,42,1.9%


## Model Selection และ Train / Test Split

| โมเดล | เหตุผลที่นำมาเปรียบเทียบ |
|---|---|
| Logistic Regression | อ่านผลได้ง่ายและใช้เป็นโมเดลหลักสำหรับการปรับเทียบคะแนน |
| Decision Tree | จับเงื่อนไขแบบกิ่งการตัดสินใจได้ |
| Random Forest | รวมผลจากต้นไม้หลายต้นเพื่อลดความผันผวน |
| Extra Trees | ใช้การสุ่มจุดแบ่งเพิ่มขึ้นเพื่อสร้างต้นไม้ที่หลากหลาย |

ทุกชุดใช้เฉพาะอุบัติเหตุในกรุงเทพมหานคร **ปี 2020–2024** ใช้ฝึกโมเดล **ปี 2025** ใช้เปรียบเทียบเพื่อเลือกโมเดลหลักด้วย Balanced Accuracy และปรับเทียบคะแนนของโมเดลที่เลือก **ปี 2026** ใช้รายงานผลทดสอบข้ามปี ชุดฝึกมีเหตุเสียชีวิตน้อยกว่า จึงใช้ `class_weight='balanced'` หรือ `balanced_subsample` ตามชนิดโมเดล และใช้ DummyClassifier เป็นค่าพื้นฐานสำหรับเทียบผล

ขั้นตอนเตรียมข้อมูลใน Pipeline แปลงรหัสสายทางและสภาพอากาศด้วย One-Hot Encoding ส่วนตัวแปรตัวเลขเติมค่าว่างด้วยมัธยฐานและปรับสเกลจากชุดฝึก

## Classification Models — ฝึกและเลือกโมเดลจากผลปี 2025

### กำหนดขั้นตอนเตรียมข้อมูลและโมเดล

In [107]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# One-Hot แปลงหมวดหมู่เป็นช่อง 0/1 เช่น ช่อง “ฝนตก” มีค่า 1 เมื่อเหตุเกิดขณะฝนตก
categorical_steps = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=5)),
])

# เติมตัวเลขที่ขาดด้วยมัธยฐาน แล้วปรับสเกลให้ตัวแปรที่หน่วยต่างกันอยู่ในช่วงที่เทียบกันง่ายขึ้น
numeric_steps = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler(with_mean=False)),
])

# เลือกวิธีเตรียมข้อมูลตามชนิดคอลัมน์ แล้วรวมกลับเป็นข้อมูลชุดเดียวก่อนเข้าโมเดล
preprocessor = ColumnTransformer([
    ('categorical', categorical_steps, CATEGORICAL_FEATURES),
    ('numeric', numeric_steps, NUMERIC_FEATURES),
])

# Baseline ทายคลาสที่พบบ่อยที่สุด ใช้เป็นจุดอ้างอิงเทียบกับโมเดลที่ฝึกจริงทั้ง 4 ตัว
# class_weight ช่วยให้โมเดลให้ความสำคัญกับเหตุเสียชีวิตซึ่งมีน้อยในชุดฝึก
models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(
        max_iter=1200, class_weight='balanced', solver='liblinear', random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=20, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=180, min_samples_leaf=3,
        class_weight='balanced_subsample', n_jobs=-1, random_state=42
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=180, min_samples_leaf=3,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
}

### ฝึกด้วยปี 2020–2024 และเปรียบเทียบด้วยปี 2025

In [108]:
# ฝึกทุก Pipeline ด้วยปี 2020–2024 แล้วใช้ปี 2025 เปรียบเทียบก่อนรายงานผลปี 2026
train_mask = df['year'].between(train_start_year, train_end_year)
validation_mask = df['year'] == validation_year
# X คือข้อมูลที่ให้โมเดลอ่าน ส่วน y คือคำตอบจริงว่าแต่ละเหตุมีผู้เสียชีวิตหรือไม่
X_train = df.loc[train_mask, FEATURES]
y_train = df.loc[train_mask, 'fatal']
X_validation = df.loc[validation_mask, FEATURES]
y_validation = df.loc[validation_mask, 'fatal']

fitted_models = {}
validation_rows = []
for name, estimator in models.items():
    # clone ให้แต่ละโมเดลได้ Pipeline ใหม่ ตัวเติมค่าว่างและตัวแปลงข้อมูลจึงเรียนจากชุดฝึกเท่านั้น
    pipeline = Pipeline([('preprocess', clone(preprocessor)), ('model', clone(estimator))])
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline
    # ทายปี 2025 ด้วยโมเดลที่ฝึกเสร็จแล้ว จากนั้นเทียบกับคำตอบจริงของปีเดียวกัน
    prediction = pipeline.predict(X_validation)
    validation_rows.append({
        'Model': name,
        'Balanced Accuracy': balanced_accuracy_score(y_validation, prediction),
        'Recall เสียชีวิต': recall_score(y_validation, prediction, zero_division=0),
    })

validation_results = pd.DataFrame(validation_rows).sort_values('Balanced Accuracy', ascending=False)
# แปลงเป็นข้อความเปอร์เซ็นต์เฉพาะตารางแสดงผล เพื่อเก็บค่าตัวเลขไว้ใช้เรียงอันดับ
validation_display = validation_results.copy()
for column in ['Balanced Accuracy', 'Recall เสียชีวิต']:
    validation_display[column] = validation_display[column].map(lambda value: f'{value:.1%}')
display(validation_display.reset_index(drop=True))

logistic_validation = validation_results.set_index('Model').loc['Logistic Regression']
validation_leader = validation_results.iloc[0]
if validation_leader['Model'] == 'Logistic Regression':
    selection_reason = (
        f"เลือก **Logistic Regression** เพราะได้ Balanced Accuracy สูงสุดในปี 2025 "
        f"ที่ **{logistic_validation['Balanced Accuracy']:.1%}** "
        f"และตรวจพบเหตุเสียชีวิต **{logistic_validation['Recall เสียชีวิต']:.1%}**"
    )
else:
    selection_reason = (
        f"ปี 2025 **{validation_leader['Model']}** ได้ Balanced Accuracy สูงสุด "
        f"**{validation_leader['Balanced Accuracy']:.1%}** "
        f"ส่วน Logistic Regression ซึ่งเป็นโมเดลหลักของ Notebook ได้ **{logistic_validation['Balanced Accuracy']:.1%}**"
    )
display(Markdown(selection_reason + ' Balanced Accuracy ให้ความสำคัญกับการทายถูกของทั้งสองคลาสเท่ากัน'))

,Model,Balanced Accuracy,Recall เสียชีวิต
0,Logistic Regression,64.9%,59.3%
1,Decision Tree,58.3%,39.0%
2,Extra Trees,56.4%,16.9%
3,Random Forest,53.4%,8.5%
4,Baseline,50.0%,0.0%


เลือก **Logistic Regression** เพราะได้ Balanced Accuracy สูงสุดในปี 2025 ที่ **64.9%** และตรวจพบเหตุเสียชีวิต **59.3%** Balanced Accuracy ให้ความสำคัญกับการทายถูกของทั้งสองคลาสเท่ากัน

### ผลทดสอบปี 2026

วัดผลของทุกโมเดลบนอุบัติเหตุในกรุงเทพมหานครชุดเดียวกันในช่วง **1 ม.ค.–31 ต.ค. 2026** เพื่อเปรียบเทียบประสิทธิภาพข้ามปี

แถว **Baseline (DummyClassifier)** เป็นตัวอ้างอิงที่ทายคลาสที่พบบ่อยที่สุด กราฟด้านล่างเปรียบเทียบ Logistic Regression, Decision Tree, Random Forest และ Extra Trees


In [109]:
# รายงานผลของโมเดลที่ฝึกเสร็จแล้วบนอุบัติเหตุปี 2026 ชุดเดียวกัน
test_mask = df['year'] == test_year
X_test = df.loc[test_mask, FEATURES]
y_test = df.loc[test_mask, 'fatal']

rows = []
for name, pipeline in fitted_models.items():
    prediction = pipeline.predict(X_test)
    # predict ให้คลาส 0/1 สำหรับป้ายเตือน ส่วนคอลัมน์ 1 ของ predict_proba คือคะแนนคลาสเสียชีวิต
    probability = pipeline.predict_proba(X_test)[:, 1]
    # อ่านตารางจากซ้ายไปขวา: TN = ไม่เตือนถูก, FP = เตือนเกิน, FN = พลาด, TP = เตือนถูก
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, prediction),
        'Balanced Accuracy': balanced_accuracy_score(y_test, prediction),
        'Precision เสียชีวิต': precision_score(y_test, prediction, zero_division=0),
        'Recall เสียชีวิต': recall_score(y_test, prediction),
        'F1 เสียชีวิต': f1_score(y_test, prediction),
        'ROC AUC': roc_auc_score(y_test, probability),
        'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
    })

results = pd.DataFrame(rows).sort_values('Balanced Accuracy', ascending=False)

# แสดงผลโดยไม่ใช้ DataFrame.style เพื่อให้ทำงานได้แม้ไม่มี Jinja2
results_display = results.copy()
percentage_columns = [
    'Accuracy',
    'Balanced Accuracy',
    'Precision เสียชีวิต',
    'Recall เสียชีวิต',
    'F1 เสียชีวิต',
    'ROC AUC',
]
for column in percentage_columns:
    results_display[column] = results_display[column].map(lambda value: f'{value:.1%}')
display(results_display.reset_index(drop=True))

# อธิบายค่าอ้างอิงไว้กับตาราง เพราะกราฟด้านล่างแสดงเฉพาะสี่โมเดลที่เปรียบเทียบ
baseline = results.set_index('Model').loc['Baseline']
display(Markdown(
    f"**Baseline (DummyClassifier)** ทายว่าไม่มีผู้เสียชีวิตทุกเหตุ จึงได้ Accuracy "
    f"**{baseline['Accuracy']:.1%}** และตรวจพบเหตุเสียชีวิต **{baseline['Recall เสียชีวิต']:.1%}**"
))


,Model,Accuracy,Balanced Accuracy,Precision เสียชีวิต,Recall เสียชีวิต,F1 เสียชีวิต,ROC AUC,TP,FN,FP,TN
0,Logistic Regression,74.6%,66.0%,4.2%,57.1%,7.8%,72.5%,24,18,548,1638
1,Extra Trees,94.9%,60.1%,11.0%,23.8%,15.0%,76.5%,10,32,81,2105
2,Decision Tree,78.2%,57.4%,3.2%,35.7%,5.8%,64.0%,15,27,459,1727
3,Random Forest,96.6%,56.2%,13.0%,14.3%,13.6%,76.1%,6,36,40,2146
4,Baseline,98.1%,50.0%,0.0%,0.0%,0.0%,50.0%,0,42,0,2186


**Baseline (DummyClassifier)** ทายว่าไม่มีผู้เสียชีวิตทุกเหตุ จึงได้ Accuracy **98.1%** และตรวจพบเหตุเสียชีวิต **0.0%**

### เปรียบเทียบประสิทธิภาพบนชุดทดสอบปี 2026

Precision, Recall และ F1 วัดเฉพาะกลุ่มเหตุที่มีผู้เสียชีวิต ส่วน ROC-AUC วัดว่าโมเดลจัดอันดับเหตุเสียชีวิตได้ดีเพียงใดเมื่อพิจารณาหลายเกณฑ์แจ้งเตือน ค่า Balanced Accuracy อยู่ในตารางผลด้านบนเพื่อเทียบความถูกต้องของทั้งสองคลาส

In [110]:
# กราฟเน้นโมเดลที่ฝึกจริงทั้ง 4 ตัว โดยยังดู Baseline ได้ในตารางผล
metric_columns = ['Precision เสียชีวิต', 'Recall เสียชีวิต', 'F1 เสียชีวิต', 'ROC AUC']
model_order = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Extra Trees']
# เลือกชื่อโมเดลและคอลัมน์ตามลำดับที่ต้องการให้ปรากฏบนกราฟ
chart_results = results.set_index('Model').loc[model_order]
comparison_plot = chart_results[metric_columns].copy()
comparison_plot.columns = ['Precision', 'Recall', 'F1-score', 'ROC-AUC']
show_grouped_bars(
    'Comparison of Classification Models (Bangkok, 2026)',
    comparison_plot, np.linspace(0, 1, 6), 'Model', 'Score',
    ['#4c72b0', '#dd8452', '#55a868', '#c44e52'],
)

# สรุปเฉพาะโมเดลในกราฟ โดยอ่าน Balanced Accuracy เพิ่มจากตาราง
best_balance = chart_results['Balanced Accuracy'].idxmax()
best_recall = chart_results['Recall เสียชีวิต'].idxmax()
best_f1 = chart_results['F1 เสียชีวิต'].idxmax()
best_auc = chart_results['ROC AUC'].idxmax()
display(Markdown(
    f"- **{best_balance}** ได้ Balanced Accuracy สูงสุดในตาราง **{chart_results.loc[best_balance, 'Balanced Accuracy']:.1%}**\n"
    f"- **{best_recall}** ได้ Recall สูงสุด **{chart_results.loc[best_recall, 'Recall เสียชีวิต']:.1%}** "
    f"โดยมี Precision **{chart_results.loc[best_recall, 'Precision เสียชีวิต']:.1%}**\n"
    f"- **{best_f1}** ได้ F1 สูงสุด **{chart_results.loc[best_f1, 'F1 เสียชีวิต']:.1%}**\n"
    f"- **{best_auc}** ได้ ROC-AUC สูงสุด **{chart_results.loc[best_auc, 'ROC AUC']:.1%}**"
))


- **Logistic Regression** ได้ Balanced Accuracy สูงสุดในตาราง **66.0%**
- **Logistic Regression** ได้ Recall สูงสุด **57.1%** โดยมี Precision **4.2%**
- **Extra Trees** ได้ F1 สูงสุด **15.0%**
- **Extra Trees** ได้ ROC-AUC สูงสุด **76.5%**

## Logistic Regression — โมเดลหลัก

กราฟ Confusion Matrix แสดงผลทดสอบปี 2026 ของโมเดลหลักที่เลือกจากผลปี 2025 โดยใช้เกณฑ์แจ้งเตือนเดิม แถวคือผลจริง คอลัมน์คือผลที่ทำนาย สีเข้มหมายถึงจำนวนเหตุมาก

In [111]:
# ดึงจำนวนจากผลทดสอบปี 2026 ของโมเดลหลัก แล้วส่งให้ฟังก์ชันวาดตารางสี
logistic = results.set_index('Model').loc['Logistic Regression']
counts = ((int(logistic['TN']), int(logistic['FP'])), (int(logistic['FN']), int(logistic['TP'])))
show_confusion_matrix(counts)

display(Markdown(
    f"**อ่านกราฟ** โมเดลตรวจพบเหตุเสียชีวิต **{int(logistic['TP']):,} เหตุ** "
    f"พลาด **{int(logistic['FN']):,} เหตุ** และเตือนเกิน **{int(logistic['FP']):,} เหตุ**"
))

**อ่านกราฟ** โมเดลตรวจพบเหตุเสียชีวิต **24 เหตุ** พลาด **18 เหตุ** และเตือนเกิน **548 เหตุ**

### ปรับเทียบความน่าจะเป็นด้วยข้อมูลปี 2025

ให้โมเดลเดิมคำนวณคะแนนอุบัติเหตุปี 2025 แล้วใช้ผลจริงของปีเดียวกันหาสูตรปรับคะแนนแบบ sigmoid ตารางด้านล่างเทียบคะแนนก่อนและหลังปรับกับอัตราเหตุเสียชีวิตจริงในปี 2026 ค่า Brier score ยิ่งต่ำยิ่งดี


In [112]:
risk_model = fitted_models['Logistic Regression']

# ให้โมเดลหลักคำนวณคะแนนดิบของปี 2025 แล้วใช้ผลจริงปีเดียวกันสอนสูตร sigmoid
# reshape(-1, 1) เปลี่ยนคะแนนให้เป็นตารางหนึ่งคอลัมน์ โดยแต่ละแถวแทนอุบัติเหตุหนึ่งเหตุ
fatal_calibrator = LogisticRegression(C=1000)
fatal_calibrator.fit(
    risk_model.decision_function(X_validation).reshape(-1, 1),
    y_validation,
)

# decision_function = 0 ตรงกับเกณฑ์ 50% ของ Logistic Regression ก่อนปรับเทียบ
calibrated_alert_threshold = float(fatal_calibrator.predict_proba([[0.0]])[0, 1])

def calibrated_fatal_probability(events):
    # ให้โมเดลหลักคิดคะแนนก่อน แล้วค่อยแปลงคะแนนเป็นเปอร์เซ็นต์ที่ใช้แสดงผล
    raw_scores = risk_model.decision_function(events).reshape(-1, 1)
    return fatal_calibrator.predict_proba(raw_scores)[:, 1]

# ใช้ผลจริงของปี 2026 วัด Brier score ของทั้งสองชุดคะแนน ค่ายิ่งต่ำยิ่งดี
original_test_scores = risk_model.predict_proba(X_test)[:, 1]
calibrated_test_scores = calibrated_fatal_probability(X_test)
print(f'อัตราเหตุเสียชีวิตจริงในกรุงเทพฯ ปี 2026: {y_test.mean():.1%}')
display(pd.DataFrame({
    'คะแนน': ['ก่อนปรับเทียบ', 'หลังปรับเทียบ'],
    'คะแนนเฉลี่ยปี 2026': [
        f'{original_test_scores.mean():.1%}',
        f'{calibrated_test_scores.mean():.1%}',
    ],
    'Brier score': [
        f'{brier_score_loss(y_test, original_test_scores):.4f}',
        f'{brier_score_loss(y_test, calibrated_test_scores):.4f}',
    ],
}))

อัตราเหตุเสียชีวิตจริงในกรุงเทพฯ ปี 2026: 1.9%


,คะแนน,คะแนนเฉลี่ยปี 2026,Brier score
0,ก่อนปรับเทียบ,36.6%,0.1713
1,หลังปรับเทียบ,2.0%,0.0176


In [113]:
# เกณฑ์นี้ใช้กับคะแนนรายเหตุ ตารางนำมาเทียบกับสัดส่วนจริงรายปีเพื่อเป็นจุดอ้างอิง
annual_comparison = yearly[['fatal_rate']].copy()
annual_comparison['difference'] = (annual_comparison['fatal_rate'] - calibrated_alert_threshold) * 100
annual_comparison['สัดส่วนเหตุเสียชีวิตจริง'] = annual_comparison['fatal_rate'].map(lambda rate: f'{rate:.1%}')
annual_comparison['ส่วนต่างจากเกณฑ์ (จุดเปอร์เซ็นต์)'] = annual_comparison['difference'].map(lambda delta: f'{delta:+.2f}')
print(f'เกณฑ์แจ้งเตือนบนคะแนนหลังปรับเทียบ {calibrated_alert_threshold:.2%}')
display(annual_comparison[['สัดส่วนเหตุเสียชีวิตจริง', 'ส่วนต่างจากเกณฑ์ (จุดเปอร์เซ็นต์)']])

เกณฑ์แจ้งเตือนบนคะแนนหลังปรับเทียบ 2.37%


,สัดส่วนเหตุเสียชีวิตจริง,ส่วนต่างจากเกณฑ์ (จุดเปอร์เซ็นต์)
year,,
2020,1.4%,-0.95
2021,2.3%,-0.03
2022,1.5%,-0.86
2023,2.5%,+0.14
2024,1.5%,-0.83
2025,2.1%,-0.31
2026,1.9%,-0.48


## Prediction — ทดลองทำนายด้วยข้อมูล 4 ช่อง

กรอกกิโลเมตรของจุดที่อยู่ในกรุงเทพมหานคร และใช้รหัสสายทางกับสภาพอากาศที่มีในชุดฝึก หากกรอกค่าที่โมเดลไม่รู้จัก เซลล์จะแจ้งว่าค่าใดไม่พบในชุดฝึก

In [114]:
# แก้เฉพาะค่า 4 บรรทัดนี้ แล้วกด Run เซลล์
route_code = '35'
kilometer = 8.3
event_datetime = '2026-01-02 01:40'
weather = 'แจ่มใส'

# ตรวจวันที่ก่อนใช้แยกเดือน วันในสัปดาห์ และชั่วโมง
kilometer = float(kilometer)
event_dt = pd.to_datetime(event_datetime, errors='coerce')
if pd.isna(event_dt):
    raise ValueError('กรุณากรอกวันที่และเวลาเป็น YYYY-MM-DD HH:MM')

# ตรวจว่ารหัสสายทางและสภาพอากาศอยู่ในชุดฝึก เพื่อแจ้งผู้ใช้ได้ตรงช่องเมื่อกรอกผิด
known_routes = set(X_train['รหัสสายทาง'].dropna().astype(str))
known_weather = set(X_train['สภาพอากาศ'].dropna().astype(str))
if route_code not in known_routes:
    raise ValueError(f'ไม่พบรหัสสายทาง {route_code} ในชุดฝึก')
if weather not in known_weather:
    raise ValueError(f'ไม่พบสภาพอากาศ {weather} ในชุดฝึก')

# สร้างข้อมูลหนึ่งแถวและเรียงคอลัมน์ตาม FEATURES แบบเดียวกับตอนฝึก
new_event = pd.DataFrame([{
    'รหัสสายทาง': route_code,
    'สภาพอากาศ': weather,
    'ก.ม.': kilometer,
    'month': int(event_dt.month),
    'weekday': int(event_dt.dayofweek),
    'hour': float(event_dt.hour + event_dt.minute / 60),
}])[FEATURES]

# เปอร์เซ็นต์ใช้คะแนนหลังปรับเทียบ ส่วนป้ายเตือนใช้คลาสจากโมเดลเดิม
fatal_probability = float(calibrated_fatal_probability(new_event)[0])
predicted_class = int(risk_model.predict(new_event)[0])
result_label = 'เสี่ยงรุนแรง' if predicted_class == 1 else 'ความเสี่ยงทั่วไป'

# ประวัติใช้ช่วงหนึ่งกิโลเมตร เช่น 8.3 อยู่ในช่วงตั้งแต่ กม. 8 ถึงก่อน กม. 9
segment_start = int(np.floor(kilometer))
segment_end = segment_start + 1
same_route = df['รหัสสายทาง'] == route_code
within_segment = df['ก.ม.'].between(segment_start, segment_end, inclusive='left')
# นับประวัติจากทุกปีที่มีข้อมูลก่อนปีทดสอบ
history_period = df['year'].between(train_start_year, validation_year)
history = df.loc[history_period & same_route & within_segment]
historical_accidents = int(len(history))
historical_fatal = int(history['fatal'].sum())
print(f'ถนนสาย {route_code} กม. {segment_start}–{segment_end} ({target_province})')
# อัตราย้อนหลังคำนวณได้เมื่อช่วงถนนนี้มีเหตุที่บันทึกไว้
if historical_accidents:
    historical_fatal_rate = historical_fatal / historical_accidents
    print(f'เกิดอุบัติเหตุย้อนหลัง {historical_accidents:,} ครั้ง (ปี {train_start_year}–{validation_year})')
    print(f'เป็นเหตุเสียชีวิต {historical_fatal:,} ครั้ง')
    print(f'อัตราเหตุเสียชีวิต {historical_fatal_rate:.1%}')
else:
    print(f'ไม่มีข้อมูลย้อนหลังสำหรับช่วงถนนนี้ในปี {train_start_year}–{validation_year}')
print(f'วันที่และเวลา: {event_dt:%Y-%m-%d %H:%M}')
print(f'สภาพอากาศ: {weather}')
print(f'โอกาสโดยประมาณที่เหตุจะมีผู้เสียชีวิต หากเกิดอุบัติเหตุ: {fatal_probability:.1%}')
print(f'ผลประเมิน: {result_label}')

ถนนสาย 35 กม. 8–9 (กรุงเทพมหานคร)
เกิดอุบัติเหตุย้อนหลัง 90 ครั้ง (ปี 2020–2025)
เป็นเหตุเสียชีวิต 2 ครั้ง
อัตราเหตุเสียชีวิต 2.2%
วันที่และเวลา: 2026-01-02 01:40
สภาพอากาศ: แจ่มใส
โอกาสโดยประมาณที่เหตุจะมีผู้เสียชีวิต หากเกิดอุบัติเหตุ: 4.9%
ผลประเมิน: เสี่ยงรุนแรง


## Model Export — บันทึกโมเดลเพื่อนำไปใช้ต่อ

ไฟล์นี้ระบุพื้นที่กรุงเทพมหานครไว้ใน `province` และเก็บโมเดลใน `model` และตัวปรับเทียบใน `calibrator` โดย `X` คือข้อมูลที่แปลงเป็น features ตามลำดับใน `features`

- ป้ายเตือนใช้ `model.predict(X)`
- ความน่าจะเป็นที่ปรับเทียบแล้วใช้ `calibrator.predict_proba(model.decision_function(X).reshape(-1, 1))[:, 1]`

ดูชื่อไฟล์จากผลลัพธ์ของเซลล์บันทึก หากเขียนทับไฟล์เดิมไม่ได้ Notebook จะแจ้งชื่อไฟล์ใหม่ Colab จะดาวน์โหลดไฟล์หลังบันทึกเสร็จ

In [115]:
from tempfile import NamedTemporaryFile

import joblib

model_filename = 'accident_road_risk_bangkok_4_inputs.joblib'
# เก็บทั้งโมเดลสำหรับตัดสินป้ายเตือนและตัวปรับเทียบสำหรับเปอร์เซ็นต์ที่แสดง
model_artifact = {
    'model': risk_model,
    'calibrator': fatal_calibrator,
    'features': FEATURES,
    'segment_size_km': 1,
    'province': target_province,
}
# เขียนไฟล์ชั่วคราวให้เสร็จก่อน เพื่อให้ไฟล์โมเดลเดิมยังเปิดได้หากบันทึกกลางทางล้มเหลว
target = Path(model_filename)
with NamedTemporaryFile(mode='wb', dir='.', prefix=f'{target.stem}_', suffix=target.suffix, delete=False) as output:
    saved_model = Path(output.name)
    try:
        joblib.dump(model_artifact, output)
    except Exception:
        output.close()
        saved_model.unlink(missing_ok=True)
        raise

try:
    saved_model.replace(target)
except PermissionError:
    # ถ้า Windows ไม่ให้แทนไฟล์เดิม ให้ใช้ไฟล์ใหม่ที่เขียนเสร็จแล้ว
    model_filename = saved_model.name
    print('เขียนทับไฟล์โมเดลเดิมไม่ได้ จึงใช้ชื่อใหม่')
print(f'บันทึกโมเดลแล้ว: {model_filename}')

try:
    from google.colab import files
    files.download(model_filename)
except ImportError:
    print('ไฟล์โมเดลอยู่ในโฟลเดอร์เดียวกับ Notebook')

บันทึกโมเดลแล้ว: accident_road_risk_bangkok_4_inputs.joblib
ไฟล์โมเดลอยู่ในโฟลเดอร์เดียวกับ Notebook


## Conclusion — สรุปและข้อจำกัด

Logistic Regression ที่ฝึกด้วยข้อมูลกรุงเทพมหานครใช้แสดงความน่าจะเป็นหลังปรับเทียบและออกป้ายเตือน การอ่านผลควรดู Recall และจำนวนเหตุที่เตือนเกินร่วมกับคะแนนรวม

- ความน่าจะเป็นหลังปรับเทียบเป็นค่าประมาณว่าหากเกิดอุบัติเหตุ จะมีผู้เสียชีวิตอย่างน้อยหนึ่งคนหรือไม่ ความแม่นยำของเปอร์เซ็นต์อาจต่างกันในแต่ละกลุ่มและแต่ละปี
- เซลล์ทำนายรับรหัสสายทางและสภาพอากาศที่พบในชุดฝึก หากไม่มีประวัติอุบัติเหตุของช่วงถนนในกรุงเทพมหานครในปี 2020–2025 จะแสดงว่า “ไม่มีข้อมูลย้อนหลัง” และยังแสดงผลทำนายจากโมเดล
- ควรใช้ผลเพื่อช่วยคัดกรองหรือจัดลำดับความเสี่ยง ไม่ควรใช้แทนการตัดสินใจของเจ้าหน้าที่